In [22]:
import pandas as pd

In [4]:
from recommender.content import (
    build_user_profile,
    recommend_content_movies
)

print("Content module imported successfully!")

Content module imported successfully!


In [5]:
from recommender.data_loader import (
    load_movie_data,
    build_movie_features,
    AVAILABLE_GENRES
)

movies, ratings = load_movie_data()

print("=== Loaded Data ===")
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Genres:", len(AVAILABLE_GENRES))

=== Loaded Data ===
Movies: (9742, 3)
Ratings: (100836, 4)
Genres: 18


In [6]:
movie_features = build_movie_features(
    movies=movies,
    ratings=ratings,
    m=10
)

print("=== Movie Features ===")
print("Shape:", movie_features.shape)

print(
    "Missing values:",
    movie_features.isnull().sum().sum()
)

print(
    "Valid genre movies:",
    (
        movie_features[AVAILABLE_GENRES]
        .sum(axis=1) > 0
    ).sum()
)

print(
    "Normalized rating range:",
    movie_features["normalized_rating"].min(),
    movie_features["normalized_rating"].max()
)

=== Movie Features ===
Shape: (9742, 25)
Missing values: 0
Valid genre movies: 9708
Normalized rating range: 0.0 1.0


In [7]:
from recommender.content import (
    build_user_profile,
    recommend_content_movies
)

selected_genres = ["Action", "Sci-Fi", "Thriller"]

user_ratings = {
    2571: 5.0,
    1: 4.5,
    2: 2.0
}

# Build the user preference profile
user_profile = build_user_profile(
    selected_genres=selected_genres,
    user_ratings=user_ratings,
    movie_features=movie_features,
    available_genres=AVAILABLE_GENRES
)

# Generate movie recommendations
recommendations = recommend_content_movies(
    user_profile=user_profile,
    movie_features=movie_features,
    available_genres=AVAILABLE_GENRES,
    watched_movie_ids=list(user_ratings.keys()),
    gamma=0.8,
    k=10
)

print("=== Movie Recommendations ===")

print(
    recommendations[
        [
            "movieId",
            "title",
            "recommendation_score"
        ]
    ].to_string(index=False)
)

print("\nRecommendation count:", len(recommendations))

print(
    "Watched movies included:",
    recommendations["movieId"].isin(
        user_ratings.keys()
    ).sum()
)

=== Movie Recommendations ===
 movieId                                                 title  recommendation_score
    2959                                     Fight Club (1999)              0.873933
     260             Star Wars: Episode IV - A New Hope (1977)              0.862576
     541                                   Blade Runner (1982)              0.858130
    1196 Star Wars: Episode V - The Empire Strikes Back (1980)              0.854626
  168252                                          Logan (2017)              0.833683
     904                                    Rear Window (1954)              0.827523
      50                            Usual Suspects, The (1995)              0.826263
    1210     Star Wars: Episode VI - Return of the Jedi (1983)              0.823922
   58559                               Dark Knight, The (2008)              0.821676
   48516                                  Departed, The (2006)              0.820168

Recommendation count: 10
Watched m

In [8]:
from recommender.collaborative import (
    build_user_item_matrix,
    calculate_item_similarities,
    calculate_cf_scores
)

user_item_matrix = build_user_item_matrix(ratings)

print("=== CF Matrix ===")
print("Shape:", user_item_matrix.shape)
print(
    "Observed ratings:",
    user_item_matrix.notna().sum().sum()
)

=== CF Matrix ===
Shape: (610, 9724)
Observed ratings: 100836


In [9]:
toy_story_similarities = calculate_item_similarities(
    movie_id=1,
    user_item_matrix=user_item_matrix
)

print("=== Item Similarity Test ===")

print(
    "Toy Story vs Jumanji:",
    toy_story_similarities.loc[2]
)

print(
    "Positive similarities:",
    (toy_story_similarities > 0).sum()
)

print(
    "Self similarity:",
    toy_story_similarities.loc[1]
)

=== Item Similarity Test ===
Toy Story vs Jumanji: 0.2202788858229442
Positive similarities: 1327
Self similarity: 0.0


In [10]:
user_ratings = {
    2571: 5.0,
    1: 4.5,
    2: 2.0
}

similarity_cache = {}

cf_scores = calculate_cf_scores(
    user_ratings=user_ratings,
    user_item_matrix=user_item_matrix,
    similarity_cache=similarity_cache
)

print("=== New User CF Scores ===")

print("Movies with valid CF scores:", cf_scores.notna().sum())

print("\nTop 10 CF scores:")
print(
    cf_scores.dropna()
    .sort_values(ascending=False)
    .head(10)
)

print("\nCached movies:", len(similarity_cache))

=== New User CF Scores ===
Movies with valid CF scores: 1306

Top 10 CF scores:
movieId
4226      0.647833
8970      0.629995
909       0.623727
122926    0.618664
58559     0.607487
235       0.596231
68157     0.594872
50        0.589903
1198      0.588900
741       0.586371
dtype: float64

Cached movies: 3


In [15]:
import importlib
import recommender.collaborative as cf_module
import numpy as np

importlib.reload(cf_module)

calculate_item_similarities = (
    cf_module.calculate_item_similarities
)

calculate_cf_scores = (
    cf_module.calculate_cf_scores
)

In [16]:
# Use the corrected CF function
similarities = calculate_item_similarities(
    movie_id=1,
    user_item_matrix=user_item_matrix
)

# Calculate independently using pandas
common_ratings = user_item_matrix[
    [1, 2]
].dropna()

raw_pearson = common_ratings[1].corr(
    common_ratings[2]
)

n = len(common_ratings)

expected_similarity = (
    raw_pearson * n / (n + 20)
    if n >= 10
    else 0.0
)

print("=== Corrected Pearson Verification ===")

print("Common users:", n)
print("Raw Pearson:", raw_pearson)
print("Expected similarity:", expected_similarity)
print("Module similarity:", similarities.loc[2])

print(
    "Values match:",
    np.isclose(
        expected_similarity,
        similarities.loc[2]
    )
)

=== Corrected Pearson Verification ===
Common users: 68
Raw Pearson: 0.3309783545424253
Expected similarity: 0.25575600123732867
Module similarity: 0.2557560012373286
Values match: True


In [17]:
similarity_cache = {}

cf_scores = calculate_cf_scores(
    user_ratings=user_ratings,
    user_item_matrix=user_item_matrix,
    similarity_cache=similarity_cache
)

print("=== Corrected CF Results ===")

print(
    "Valid CF scores:",
    cf_scores.notna().sum()
)

print("\nTop 10 CF scores:")
print(
    cf_scores.dropna()
    .sort_values(ascending=False)
    .head(10)
)

print("\nCached movies:", len(similarity_cache))

=== Corrected CF Results ===
Valid CF scores: 1306

Top 10 CF scores:
movieId
8970      0.661368
4226      0.649032
2146      0.648182
1635      0.630508
122926    0.629719
3744      0.628892
909       0.624565
136864    0.621165
6502      0.620061
235       0.613766
dtype: float64

Cached movies: 3


In [18]:
from recommender.hybrid import recommend_hybrid_movies

print("Hybrid module imported successfully!")

Hybrid module imported successfully!


In [19]:
selected_genres = ["Action", "Sci-Fi", "Thriller"]

user_ratings = {
    2571: 5.0,
    1: 4.5,
    2: 2.0
}

# Build the user genre profile
user_profile = build_user_profile(
    selected_genres=selected_genres,
    user_ratings=user_ratings,
    movie_features=movie_features,
    available_genres=AVAILABLE_GENRES
)

# Cache CF similarities
similarity_cache = {}

# Generate hybrid recommendations
hybrid_recommendations = recommend_hybrid_movies(
    user_profile=user_profile,
    user_ratings=user_ratings,
    movie_features=movie_features,
    available_genres=AVAILABLE_GENRES,
    user_item_matrix=user_item_matrix,
    similarity_cache=similarity_cache,
    gamma=0.8,
    beta=0.2,
    k=10
)

print("=== Hybrid Recommendations ===")

print(
    hybrid_recommendations[
        [
            "movieId",
            "title",
            "genre_similarity",
            "cf_score",
            "recommendation_score",
            "recommendation_source"
        ]
    ].to_string(index=False)
)

=== Hybrid Recommendations ===
 movieId                                                                          title  genre_similarity  cf_score  recommendation_score recommendation_source
    2959                                                              Fight Club (1999)          0.562940  0.416007              0.929400                Hybrid
     260                                      Star Wars: Episode IV - A New Hope (1977)          0.585024  0.318636              0.905061                Hybrid
      50                                                     Usual Suspects, The (1995)          0.325014  0.577316              0.903238                Hybrid
   58559                                                        Dark Knight, The (2008)          0.325014  0.603636              0.902160                Hybrid
    1196                          Star Wars: Episode V - The Empire Strikes Back (1980)          0.585024  0.304982              0.895290                Hybrid
     296 

In [20]:
print("=== Hybrid Recommendation Validation ===")

print(
    "Recommendation count:",
    len(hybrid_recommendations)
)

print(
    "Unique movie IDs:",
    hybrid_recommendations["movieId"].nunique()
)

print(
    "Watched movies included:",
    hybrid_recommendations["movieId"].isin(
        user_ratings.keys()
    ).sum()
)

print(
    "Movies with CF scores:",
    hybrid_recommendations["cf_score"].notna().sum()
)

print(
    "Cached movies:",
    len(similarity_cache)
)

print("\nRecommendation sources:")
print(
    hybrid_recommendations[
        "recommendation_source"
    ].value_counts()
)

=== Hybrid Recommendation Validation ===
Recommendation count: 10
Unique movie IDs: 10
Watched movies included: 0
Movies with CF scores: 10
Cached movies: 3

Recommendation sources:
recommendation_source
Hybrid    10
Name: count, dtype: int64


In [23]:
empty_profile = pd.Series(
    0.0,
    index=AVAILABLE_GENRES
)

fallback_test = recommend_hybrid_movies(
    user_profile=empty_profile,
    user_ratings={},
    movie_features=movie_features,
    available_genres=AVAILABLE_GENRES,
    user_item_matrix=user_item_matrix,
    similarity_cache={},
    gamma=0.8,
    beta=0.2,
    k=10
)

print("=== Cold Start Fallback Test ===")

print(
    fallback_test[
        [
            "movieId",
            "title",
            "recommendation_source"
        ]
    ].to_string(index=False)
)

print("\nRecommendation count:", len(fallback_test))

print(
    "Fallback movies:",
    (
        fallback_test["recommendation_source"]
        == "Popularity Fallback"
    ).sum()
)

=== Cold Start Fallback Test ===
 movieId                                                                       title recommendation_source
     318                                            Shawshank Redemption, The (1994)   Popularity Fallback
     858                                                       Godfather, The (1972)   Popularity Fallback
    2959                                                           Fight Club (1999)   Popularity Fallback
    1221                                              Godfather: Part II, The (1974)   Popularity Fallback
      50                                                  Usual Suspects, The (1995)   Popularity Fallback
     260                                   Star Wars: Episode IV - A New Hope (1977)   Popularity Fallback
     750 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)   Popularity Fallback
    1213                                                           Goodfellas (1990)   Popularity Fallback
    